# Stage B3 — Retrieval evaluation (the real test)

**Experiment B — Practical application: MobileCLIP → SigLIP 2 adapter**

Goal: one linear matrix that maps iPhone-tier MobileCLIP-S1 image
embeddings into server-tier SigLIP 2 space, so a **single Qdrant
collection** serves both tiers — the phone indexes images offline, the
server queries the same index with SigLIP text embeddings.


## What this stage does
Text-to-image retrieval on held-out images. The query is a SigLIP **text**
embedding (exactly what the server does in production). Three galleries:
- **A. SigLIP native** image embeddings → the ceiling
- **B. MobileCLIP + adapter** → our system
- **C. MobileCLIP raw** (no adapter) → lower baseline, expected ≈ 0

This is the criterion that separates "statistical correlation" from "the
information actually transfers": R² can be decent while the nuances that
drive retrieval are lost.

## Success criterion
Variant B achieves **≥ 90%** of variant A's Recall@1/5/10.
If 70-90%: upgrade the adapter to a small 1-2 layer MLP and re-run.
If < 70%: keep separate indexes per tier.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Functions
def recall_at_k(sim, ks=(1, 5, 10)):
    """sim: [n_queries, n_gallery], ground truth is the diagonal."""
    ranks = (-sim).argsort(axis=1)
    n = sim.shape[0]
    out = {}
    for k in ks:
        hits = (ranks[:, :k] == np.arange(n)[:, None]).any(1).mean()
        out[k] = hits
    return out


def l2n(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)


def main():
    pairs = np.load(str(DATA_DIR / "pairs.npz"))
    ad = np.load(str(DATA_DIR / "adapter.npz"))
    te = ad["eval_idx"]

    txt = pairs["sig_txt"][te]           # queries (server side)
    sig_img = pairs["sig_img"][te]       # ceiling gallery
    mob_img = pairs["mob_img"][te]

    W = ad["W_ridge"]
    adapted = l2n(mob_img @ W)           # our system gallery

    # pad raw mobileclip to siglip dim for the (expected-to-fail) baseline
    d_sig = sig_img.shape[1]
    raw = mob_img
    if raw.shape[1] < d_sig:
        raw = np.pad(raw, ((0, 0), (0, d_sig - raw.shape[1])))
    raw = l2n(raw[:, :d_sig])

    results = {
        "A. SigLIP native (ceiling)": recall_at_k(txt @ sig_img.T),
        "B. MobileCLIP + adapter   ": recall_at_k(txt @ adapted.T),
        "C. MobileCLIP raw (base)  ": recall_at_k(txt @ raw.T),
    }

    print(f"Held-out gallery size: {len(te)} images\n")
    print(f"{'variant':<30} R@1     R@5     R@10")
    for name, r in results.items():
        print(f"{name:<30} {r[1]:.3f}   {r[5]:.3f}   {r[10]:.3f}")

    ceil = results["A. SigLIP native (ceiling)"]
    ours = results["B. MobileCLIP + adapter   "]
    for k in (1, 5, 10):
        pct = 100 * ours[k] / max(ceil[k], 1e-9)
        verdict = "PASS" if pct >= 90 else "below target"
        print(f"\nR@{k}: adapter keeps {pct:.1f}% of ceiling -> {verdict}",
              end="")
    print()

In [ ]:
# Run the evaluation (requires pairs.npz and adapter.npz)
main()

## B3-bis — Geometry of the shared index (real embeddings)

The recall table above says the adapter works. This cell shows *what it
did to the space*, plotted from the **actual held-out embeddings** — not
an illustration.

Three point sets, all projected through one shared 2-D PCA so the two
panels are directly comparable:

| Set | What it is |
|---|---|
| server (blue) | `sig_img[eval]` — SigLIP native, the target space |
| phone raw (red) | `mob_img[eval]` zero-padded to 768 — exactly gallery C above |
| phone adapted (teal) | `L2norm(mob_img[eval] @ W)` — exactly gallery B above |

Grey lines join each image's phone vector to **its own** server vector, so
line length is the per-image translation error. The measured mean cosines
are printed first and written into the panel titles, so the figure states
its own evidence.

Runtime: seconds. Saves `geometry_real.png` to `DATA_DIR`.

In [ ]:
# B3-bis: geometry of the shared index, from real held-out embeddings
import matplotlib.pyplot as plt

TEAL, BLUE, GRAY, RED, NAVY = ("#0f766e", "#1a5276", "#95a5a6",
                               "#c0392b", "#1a1a2e")
N_SHOW = 60                      # points drawn; stats use all held-out

pairs = np.load(str(DATA_DIR / "pairs.npz"))
ad = np.load(str(DATA_DIR / "adapter.npz"))
te = ad["eval_idx"]
W = ad["W_ridge"].astype(np.float32)

sig = l2n(pairs["sig_img"][te].astype(np.float32))
mob = pairs["mob_img"][te].astype(np.float32)
adapted = l2n(mob @ W)

d_sig = sig.shape[1]
raw = np.pad(mob, ((0, 0), (0, d_sig - mob.shape[1])))
raw = l2n(raw)

cos_adapted = float((adapted * sig).sum(1).mean())
cos_raw = float((raw * sig).sum(1).mean())
print(f"MEASURED on {len(te)} held-out images:")
print(f"  adapted -> its own target, mean cosine : {cos_adapted:.3f}")
print(f"  raw     -> its own target, mean cosine : {cos_raw:.3f}")


def project(*arrays):
    """One shared 2-D PCA over the union, so panels are comparable."""
    U = np.vstack(arrays).astype(np.float64)
    U = U - U.mean(0)
    _, _, Vt = np.linalg.svd(U, full_matrices=False)
    P = U @ Vt[:2].T
    out, i = [], 0
    for a in arrays:
        out.append(P[i:i + len(a)])
        i += len(a)
    return out


idx = np.arange(min(N_SHOW, len(te)))
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5.4))

panels = [
    (axA, raw[idx], RED, "phone (raw MobileCLIP, zero-padded)",
     "A · Without the adapter", cos_raw),
    (axB, adapted[idx], TEAL, "phone (adapted: W\u00b7v)",
     "B · With the adapter", cos_adapted),
]
for ax, other, col, lab, title, c in panels:
    pm, ps = project(other, sig[idx])
    ax.scatter(*ps.T, s=28, c=BLUE, label="server (SigLIP native)",
               zorder=3)
    ax.scatter(*pm.T, s=28, c=col, marker="s", label=lab, zorder=3)
    for i in range(0, len(idx), 2):
        ax.plot([pm[i, 0], ps[i, 0]], [pm[i, 1], ps[i, 1]], c=GRAY,
                lw=0.7, alpha=0.7, zorder=1)
    ax.set_title(f"{title}\nsame-image mean cosine = {c:.3f}"
                 f"   ·   real held-out embeddings",
                 fontsize=10.5, color=NAVY)
    ax.legend(fontsize=7.6, loc="upper center", frameon=False,
              bbox_to_anchor=(0.5, -0.02))
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig(str(DATA_DIR / "geometry_real.png"), dpi=150,
            bbox_inches="tight")
plt.show()
print("\nSaved geometry_real.png")
print("Grey lines join each image's phone vector to its own server "
      "vector: line length = per-image translation error.")

## B3-ter — Optional: image-as-query (cross-tier membership / similarity)

Measures the derived capability of report Appendix D: querying the SigLIP
gallery with the **adapted phone vectors themselves** instead of caption
embeddings. R@1 here is the accuracy of "is this phone photo already in
the server library?" (exact-duplicate case: the true SigLIP twin of the
same image is the correct answer). Expected to exceed the text-query
numbers, since no modality gap is crossed. Runtime: seconds.

In [ ]:
# B3-ter: image->image retrieval across the tier boundary
pairs = np.load(str(DATA_DIR / "pairs.npz"))
ad = np.load(str(DATA_DIR / "adapter.npz"))
te = ad["eval_idx"]
W = ad["W_ridge"].astype(np.float32)

sig_gal = l2n(pairs["sig_img"][te].astype(np.float32))   # server gallery
q_adapt = l2n(pairs["mob_img"][te].astype(np.float32) @ W)  # phone queries

r_img = recall_at_k(q_adapt @ sig_gal.T)
r_txt = recall_at_k(pairs["sig_txt"][te] @ sig_gal.T)     # reference

print("image-as-query (adapted phone -> server gallery):")
print("   " + "  ".join(f"R@{k}={v:.3f}" for k, v in r_img.items()))
print("text-as-query reference (same gallery):")
print("   " + "  ".join(f"R@{k}={v:.3f}" for k, v in r_txt.items()))

# membership-gap statistic: top-1 cosine vs best wrong cosine
sims = q_adapt @ sig_gal.T
n = len(te)
true_cos = sims[np.arange(n), np.arange(n)]
sims_wrong = sims.copy()
sims_wrong[np.arange(n), np.arange(n)] = -1
best_wrong = sims_wrong.max(1)
gap = true_cos - best_wrong
print(f"\nmembership gap (true-twin cosine minus best impostor):")
print(f"   mean {gap.mean():.3f}   min {gap.min():.3f}   "
      f"fraction positive {(gap > 0).mean():.3f}")
print("a large positive gap -> a simple threshold answers")
print("'is this photo already in the server library?'")